In [ ]:
import pandas as pd
import numpy as np
np.set_printoptions(threshold=np.inf)

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from datasets import load_dataset

In [2]:
csv_data_df = pd.read_csv("../data-processing/events.csv")
csv_data_df.rename(columns={"Unnamed: 0":"ACN"}, inplace=True)
csv_data_df.head()

,ACN,assessments_primary_problem,0,1,2,3,4,5,6,7,...,14,15,16,17,18,19,20,21,22,23
0,2260174,Ambiguous,anomaly_inflight_event_/_encounter_unstabilize...,anomaly_inflight_event_/_encounter_cftt_/_cfit,anomaly_atc_issue_all_types,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_person_air_traffic_control,when_detected_in_flight,result_general_none_reported_/_taken,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2260249,Ambiguous,anomaly_aircraft_equipment_problem_critical,anomaly_inflight_event_/_encounter_cftt_/_cfit,detector_automation_aircraft_other_automation,detector_person_flight_crew,when_detected_in_flight,result_flight_crew_became_reoriented,result_flight_crew_executed_go_around_/_missed...,result_flight_crew_flc_complied_w_/_automation...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2260370,Aircraft,anomaly_aircraft_equipment_problem_critical,anomaly_deviation_/_discrepancy_-_procedural_w...,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_person_flight_crew,when_detected_in_flight,result_flight_crew_landed_in_emergency_condition,result_general_flight_cancelled_/_delayed,result_aircraft_aircraft_damaged,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2261277,Human Factors,anomaly_airspace_violation_all_types,anomaly_conflict_airborne_conflict,anomaly_deviation_/_discrepancy_-_procedural_far,anomaly_deviation_/_discrepancy_-_procedural_u...,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_person_flight_crew,when_detected_in_flight,result_general_none_reported_/_taken,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2261317,Procedure,anomaly_inflight_event_/_encounter_weather_/_t...,anomaly_deviation_-_speed_all_types,anomaly_deviation_-_track_/_heading_all_types,anomaly_inflight_event_/_encounter_loss_of_air...,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_automation_aircraft_other_automation,detector_person_flight_crew,when_detected_in_flight,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Aggregate the events into one list to be used by the onehot encoder
aggregate_events_list = csv_data_df[csv_data_df.columns[2:]].agg(list, axis=1)

# Drop NaN entries (there's probably a cleaner way to do this)
for i in range(len(aggregate_events_list)): 
    for j in range(len(aggregate_events_list[i])):
        if not isinstance(aggregate_events_list[i][j], str):
            aggregate_events_list[i] = aggregate_events_list[i][:j]
            break

aggregate_events_list.head()

0    [anomaly_inflight_event_/_encounter_unstabiliz...
1    [anomaly_aircraft_equipment_problem_critical, ...
2    [anomaly_aircraft_equipment_problem_critical, ...
3    [anomaly_airspace_violation_all_types, anomaly...
4    [anomaly_inflight_event_/_encounter_weather_/_...
dtype: object

In [4]:
# Create a new dataframe with all events aggregated into a list (to be used by the onehotter)
aggregate_events_df = pd.DataFrame(csv_data_df[csv_data_df.columns[:2]])
aggregate_events_df["events"] = aggregate_events_list
aggregate_events_df.head()

,ACN,assessments_primary_problem,events
0,2260174,Ambiguous,[anomaly_inflight_event_/_encounter_unstabiliz...
1,2260249,Ambiguous,"[anomaly_aircraft_equipment_problem_critical, ..."
2,2260370,Aircraft,"[anomaly_aircraft_equipment_problem_critical, ..."
3,2261277,Human Factors,"[anomaly_airspace_violation_all_types, anomaly..."
4,2261317,Procedure,[anomaly_inflight_event_/_encounter_weather_/_...


### One Hotting The Data

In [5]:
ohe = OneHotEncoder()   # <-- This is unnecessary
mlb = MultiLabelBinarizer()

In [9]:
onehotted = mlb.fit_transform(aggregate_events_df.events)
onehot_events_df = pd.DataFrame(onehotted, columns=mlb.classes_)

# Combine into one dataframe with the ACN and primary problem
onehot_events_df = pd.concat([aggregate_events_df[aggregate_events_df.columns[:2]], onehot_events_df], axis=1)
onehot_events_df.set_index("ACN", inplace=True)
onehot_events_df.head()

,assessments_primary_problem,anomaly_aircraft_equipment_problem_critical,anomaly_aircraft_equipment_problem_less_severe,anomaly_airspace_violation_all_types,anomaly_atc_issue_all_types,anomaly_conflict_airborne_conflict,anomaly_conflict_ground_conflict,anomaly_conflict_nmac,anomaly_critical,anomaly_deviation_-_altitude_crossing_restriction_not_met,...,when_detected_not_stated,when_detected_postflight,when_detected_preflight,when_detected_pushback,when_detected_routine_inspection,when_detected_takeoff,when_detected_takeoff_roll,when_detected_taxi,when_detected_towing,when_detected_unknown
ACN,,,,,,,,,,,,,,,,,,,,,
2260174,Ambiguous,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2260249,Ambiguous,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2260370,Aircraft,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2261277,Human Factors,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2261317,Procedure,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
# Load dataset splits
train_dataset = load_dataset("sookiemonster/asrs-narratives", split="train")
validation_dataset = load_dataset("sookiemonster/asrs-narratives", split="validation")
test_dataset = load_dataset("sookiemonster/asrs-narratives", split="test")

train_acns = set(train_dataset["acn"])
validation_acns = set(validation_dataset["acn"])
test_acns = set(test_dataset["acn"])

print(len(train_acns))
print(len(validation_acns))
print(len(test_acns))

10360
4441
9868


In [25]:
# Split up the main dataframe
train_events_df = onehot_events_df.iloc[onehot_events_df.index.isin(train_acns)]
validation_events_df = onehot_events_df.iloc[onehot_events_df.index.isin(validation_acns)]
test_events_df = onehot_events_df.iloc[onehot_events_df.index.isin(test_acns)]

# Lengths don't match because I removed empty rows from the events dataset, might be something to consider to improve the model 
print(len(train_events_df))
print(len(validation_events_df))
print(len(test_events_df))

10293
4415
9826


### Model

In [36]:
trainX = train_events_df.drop(columns="assessments_primary_problem")
trainY = train_events_df["assessments_primary_problem"]

testX = test_events_df.drop(columns="assessments_primary_problem")
testY = test_events_df["assessments_primary_problem"]

In [39]:
rfc = RandomForestClassifier()

In [46]:
rfc.fit(trainX, trainY)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [47]:
rfc.score(testX, testY)

0.653673926317932

In [ ]:
print(classification_report(testY, rfc.predict(testX)))